# Kaggle vLLM + ngrok

Run cells in order. GPU: T4 x2.

Requires Kaggle secrets: `NGROK_AUTH_TOKEN`, `HF_TOKEN`.

When done, opencode.json `baseURL` = printed NGROK URL + `/v1`.

In [1]:
%pip install -q vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.9/312.9 MB 6.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 89.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.9/184.9 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 100.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 43.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.4/358.4 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━

In [2]:
!pkill -9 -x ngrok || true
!sleep 1
!which ngrok || curl -sSL -o /tmp/ngrok.zip https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip
!unzip -q -o /tmp/ngrok.zip -d /tmp || true
!mv -f /tmp/ngrok /usr/local/bin/ngrok || true

In [3]:
!pkill -9 -if vllm || true
!pkill -9 -x ngrok || true
!fuser -k 8000/tcp 2>/dev/null || true
!sleep 2
!nvidia-smi

Wed Aug 26 09:24:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import importlib
import json
import os
import subprocess
import time
import urllib.error
import urllib.request

try:
    kaggle_secrets = importlib.import_module("kaggle_secrets")
    UserSecretsClient = kaggle_secrets.UserSecretsClient
except ModuleNotFoundError as exc:
    raise RuntimeError("This cell must be run in a Kaggle notebook.") from exc

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

subprocess.run(
    ["ngrok", "config", "add-authtoken", secrets.get_secret("NGROK_AUTH_TOKEN")],
    check=True,
)
with open("/tmp/ngrok.log", "w") as ngrok_log:
    p = subprocess.Popen(
        ["ngrok", "http", "8000", "--log", "stdout"],
        stdout=ngrok_log,
        stderr=subprocess.STDOUT,
    )

for _ in range(60):
    try:
        with urllib.request.urlopen(
            "http://127.0.0.1:4040/api/tunnels", timeout=5
        ) as response:
            d = json.load(response)
        url = d["tunnels"][0]["public_url"]
        print("NGROK URL:", url)
        break
    except (urllib.error.URLError, json.JSONDecodeError, KeyError, IndexError):
        time.sleep(2)
else:
    with open("/tmp/ngrok.log") as ngrok_log:
        print(ngrok_log.read())

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
NGROK URL: https://985b-136-107-173-62.ngrok-free.app


In [ ]:
MODEL = os.environ.get("VLLM_MODEL", "Qwen/Qwen2.5-7B-Instruct-AWQ")
TP = int(os.environ.get("VLLM_TP_SIZE", "2"))

subprocess.run(
    [
        "vllm",
        "serve",
        MODEL,
        "--served-model-name",
        MODEL,
        "--tensor-parallel-size",
        str(TP),
        "--max-model-len",
        "32768",
        "--gpu-memory-utilization",
        "0.90",
        "--enforce-eager",
        "--enable-auto-tool-choice",
        "--tool-call-parser",
        "hermes",
        "--port",
        "8000",
    ],
    check=True,
)

(APIServer pid=252) INFO 08-26 09:25:04 [api_utils.py:345] 
(APIServer pid=252) INFO 08-26 09:25:04 [api_utils.py:345]        █     █     █▄   ▄█
(APIServer pid=252) INFO 08-26 09:25:04 [api_utils.py:345]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.27.1
(APIServer pid=252) INFO 08-26 09:25:04 [api_utils.py:345]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-7B-Instruct-AWQ
(APIServer pid=252) INFO 08-26 09:25:04 [api_utils.py:345]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=252) INFO 08-26 09:25:04 [api_utils.py:345] 
(APIServer pid=252) INFO 08-26 09:25:04 [api_utils.py:273] non-default args: {'model_tag': 'Qwen/Qwen2.5-7B-Instruct-AWQ', 'enable_auto_tool_choice': True, 'tool_call_parser': 'hermes', 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ', 'max_model_len': 32768, 'enforce_eager': True, 'served_model_name': ['Qwen/Qwen2.5-7B-Instruct-AWQ'], 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.9}
(APIServer pid=252) INFO 08-26 09:25:20 [model.py:645] Resolved architecture: Qwen2ForCausalLM


Parse safetensors files: 100%|██████████| 2/2 [00:00<00:00,  5.11it/s]


(APIServer pid=252) WARNING 08-26 09:25:21 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(APIServer pid=252) WARNING 08-26 09:25:21 [vllm.py:1247] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(APIServer pid=252) INFO 08-26 09:25:21 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
(APIServer pid=252) INFO 08-26 09:25:21 [vllm.py:1426] Cudagraph is disabled under eager mode
(APIServer pid=252) INFO 08-26 09:25:21 [compilation.py:329] Enabled custom fusions: norm_quant, act_quant
(EngineCore pid=338) INFO 08-26 09:25:41 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='Qwen/Qwen2.5-7B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct-AWQ', sk

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:01<00:01,  1.02s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.60it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.46it/s]
(Worker_TP0 pid=378) 


(Worker_TP0 pid=378) INFO 08-26 09:26:39 [default_loader.py:430] Loading weights took 1.39 seconds
(Worker_TP0 pid=378) INFO 08-26 09:26:41 [model_runner.py:329] Model loading took 2.67 GiB and 31.084005 seconds
(Worker_TP0 pid=378) WARNING 08-26 09:26:41 [topk_topp_sampler.py:69] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(Worker_TP1 pid=379) INFO 08-26 09:26:42 [model_runner.py:329] Model loading took 2.67 GiB and 31.365724 seconds
(Worker_TP0 pid=378) INFO 08-26 09:26:52 [gpu_worker.py:563] Available KV cache memory: 10.04 GiB
(EngineCore pid=338) INFO 08-26 09:26:52 [kv_cache_utils.py:2235] GPU KV cache size: 375,808 tokens
(EngineCore pid=338) INFO 08-26 09:26:52 [kv_cache_utils.py:2236] Maximum concurrency for 32,768 tokens per request: 11.47x
(Worker_TP0 pid=378) INFO 08-26 09:26:52 [gpu_worker.py:789] Free memory on device (14.32/14.56 GiB) on startup. Desired GPU memory utilizatio

(APIServer pid=252) INFO:     Started server process [252]
(APIServer pid=252) INFO:     Waiting for application startup.
(APIServer pid=252) INFO:     Application startup complete.


(APIServer pid=252) INFO 08-26 09:27:16 [launcher.py:105] API server: HTTP server started
(APIServer pid=252) INFO:     123.20.210.94:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=252) INFO:     123.20.210.94:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(Worker_TP0 pid=378) WARNING 08-26 09:30:56 [jit_monitor.py:135] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.
(APIServer pid=252) INFO 08-26 09:31:06 [loggers.py:310] Engine 000: Avg prompt throughput: 137.6 tokens/s, Avg generation throughput: 0.2 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 1.6%, Prefix cache hit rate: 0.0%
(APIServer pid=252) INFO 08-26 09:31:16 [loggers.py:310] Engine 000: Avg prompt throughput: 829.2 tokens/s, Avg generation throughput: 5.9 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 2.2%, Prefix cache hit rate: 0.0%
(APIServer pid=252) INFO:     